# 03 — ML Models (LightGBM + XGBoost) on the Unified Feature Table

Both ML models share the **same** feature columns, the **same** target convention (predict `weekly_units` of week W using only past info), and the **same** two-stage architecture:

- **Stage 1** — occurrence classifier `P(weekly_units > 0)`, with class-imbalance correction.
- **Stage 2** — magnitude regressor on `log1p(weekly_units)`, trained on positive rows only.
- **Final prediction** — `y_pred = P * E[units|positive]` (a **soft** combination, no hard gate).

Why soft instead of hard gate? With ~78% of validation rows being true zeros, a "predict all zeros" baseline already achieves WAPE = 1.0; any hard threshold that fires non-trivial predictions tends to over-shoot in WAPE because the magnitude regressor (trained only on positives) cannot output zero. The soft combination `P * mag` smoothly down-weights low-probability pairs and produces a **non-zero expected demand** for every (facility, product) row — which is exactly what the shopping list needs.

**Inputs:** `unified_features.parquet`.
**Outputs:** `data/lgbm_predictions.csv`, `data/xgb_predictions.csv`.


## Step 0 — Imports and split

In [5]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from google.colab import files

warnings.filterwarnings('ignore')
pd.set_option('display.width', 140)

file_paths = list(Path('/content').rglob('unified_features.parquet'))

if not file_paths:
    print("File not found. Please upload 'unified_features.parquet':")
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("Upload cancelled or failed.")
    file_paths = list(Path('/content').rglob('unified_features.parquet'))

file_path = file_paths[0]
DATA_DIR = file_path.parent

PAIR_COLS = ['NEST_NAME', 'HEALTH_FACILITY_NAME', 'PRODUCT_NAME']

uni = pd.read_parquet(file_path)
print('Unified features shape:', uni.shape)
print('Per-nest x split row counts:')
print(uni.groupby(['NEST_NAME', 'split']).size().unstack(fill_value=0))

File not found. Please upload 'unified_features.parquet':


Saving unified_features.parquet to unified_features.parquet
Unified features shape: (687678, 48)
Per-nest x split row counts:
split         test   train  valid
NEST_NAME                        
GH1 Omenako   4240   85648   4240
GH3 Vobsi     9530  192506   9530
RW1 Muhanga  17770  209686  17770
RW2 Kayonza   9910  116938   9910


## Step 1 — Define the unified feature column list

In [6]:
FEATURES = [
    # calendar
    'week_of_year', 'month', 'quarter', 'week_sin', 'week_cos', 'rainy_season',
    # lags
    'lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_8', 'lag_12',
    # rollings
    'roll_mean_4', 'roll_std_4', 'roll_max_4', 'roll_nz_rate_4',
    'roll_mean_8', 'roll_std_8', 'roll_max_8', 'roll_nz_rate_8',
    'roll_mean_13', 'roll_std_13', 'roll_max_13', 'roll_nz_rate_13',
    # intermittent
    'weeks_since_last_demand', 'last_nonzero_units',
    'historical_nonzero_rate', 'cumulative_active_weeks',
    # expanding history
    'facility_hist_mean', 'facility_hist_nz_rate',
    'product_hist_mean',  'product_hist_nz_rate',
    'fp_hist_mean',       'fp_hist_nz_rate',
    # pair-level numerics
    'adi', 'cv2', 'mean_nonzero', 'nonzero_weeks',
    # encodings
    'HEALTH_FACILITY_NAME_enc', 'PRODUCT_NAME_enc',
]
print(f'{len(FEATURES)} unified features.')

train = uni[uni['split'] == 'train'].dropna(subset=['lag_1']).reset_index(drop=True)
val   = uni[uni['split'] == 'valid'].reset_index(drop=True)
test  = uni[uni['split'] == 'test' ].reset_index(drop=True)

X_tr = train[FEATURES].fillna(-999); y_tr = train['weekly_units'].astype(float)
X_va = val[FEATURES].fillna(-999);   y_va = val['weekly_units'].astype(float)
X_te = test[FEATURES].fillna(-999);  y_te = test['weekly_units'].astype(float)

occ_tr = (y_tr > 0).astype(int)
occ_va = (y_va > 0).astype(int)
n_pos_tr = int(occ_tr.sum()); n_neg_tr = len(occ_tr) - n_pos_tr
spw = n_neg_tr / max(n_pos_tr, 1)
pos_mask_tr = (y_tr > 0).to_numpy()
pos_mask_va = (y_va > 0).to_numpy()
print(f'Rows  train={len(train):,}  valid={len(val):,}  test={len(test):,}')
print(f'Train occurrence balance: pos={n_pos_tr:,}  neg={n_neg_tr:,}  scale_pos_weight={spw:.2f}')


40 unified features.
Rows  train=596,488  valid=41,450  test=41,450
Train occurrence balance: pos=103,709  neg=492,779  scale_pos_weight=4.75


## Step 2 — Helper functions: predictions + metrics

- **Soft** `y_pred = P × mag` — smoothly weights every row by occurrence probability.
- **Hard** `y_pred = (P ≥ threshold) × mag : 0` — zero unless the classifier fires.

Extra metrics beyond MAE/WAPE:

- **Asym-WAPE** — penalises under-prediction 3× more than over-prediction (supply-chain asymmetry).
- **UPR** — under-prediction ratio: fraction of true demand that we under-served.
- **OPR** — over-prediction ratio: fraction of predicted demand that exceeds truth.

In [7]:
UNDER_PENALTY = 3.0  # under-prediction penalised 3x harder (prefer over-ordering in supply chains)

def soft_predict(clf, reg, X):
    prob = clf.predict_proba(X)[:, 1]
    mag  = np.expm1(reg.predict(X)).clip(min=0)
    return np.clip(prob * mag, 0, None), prob, mag


def hard_predict(clf, reg, X, threshold):
    prob = clf.predict_proba(X)[:, 1]
    mag  = np.expm1(reg.predict(X)).clip(min=0)
    pred = np.where(prob >= threshold, mag, 0.0)
    return pred, prob, mag


def wape(y, p):
    y = np.asarray(y, dtype=float); p = np.asarray(p, dtype=float)
    return float(np.sum(np.abs(p - y)) / max(np.sum(np.abs(y)), 1e-9))


def mae(y, p):
    return float(np.mean(np.abs(np.asarray(p, dtype=float) - np.asarray(y, dtype=float))))


def asym_wape(y, p, penalty=UNDER_PENALTY):
    y = np.asarray(y, dtype=float); p = np.asarray(p, dtype=float)
    err = y - p  # positive = under-prediction
    loss = np.where(err > 0, penalty * err, -err)
    denom = penalty * max(float(np.sum(np.abs(y))), 1e-9)
    return float(np.sum(loss) / denom)


def upr_opr(y, p):
    y = np.asarray(y, dtype=float); p = np.asarray(p, dtype=float)
    under_vol = float(np.sum(np.maximum(y - p, 0)))
    over_vol  = float(np.sum(np.maximum(p - y, 0)))
    total_act = max(float(np.sum(y)), 1e-9)
    return under_vol / total_act, over_vol / total_act


## Step 3 — LightGBM — Stage 1 (occurrence classifier)

In [8]:
lgbm_clf = lgb.LGBMClassifier(
    n_estimators=400, learning_rate=0.05, num_leaves=31,
    min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=spw, random_state=42, verbosity=-1,
)
lgbm_clf.fit(X_tr, occ_tr, eval_set=[(X_va, occ_va)],
             eval_metric='auc', callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
print(f'LGBM Stage-1 best iteration: {lgbm_clf.best_iteration_}')


Training until validation scores don't improve for 30 rounds
Early stopping, best iteration is:
[3]	valid_0's auc: 0.689433	valid_0's binary_logloss: 0.447699
LGBM Stage-1 best iteration: 3


## Step 4 — LightGBM — Stage 2 (magnitude regressor on positive rows)

In [9]:
lgbm_reg = lgb.LGBMRegressor(
    n_estimators=600, learning_rate=0.05, num_leaves=31,
    min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
    random_state=42, verbosity=-1,
)
lgbm_reg.fit(X_tr[pos_mask_tr], np.log1p(y_tr[pos_mask_tr]),
             eval_set=[(X_va[pos_mask_va], np.log1p(y_va[pos_mask_va]))],
             eval_metric='rmse',
             callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
print(f'LGBM Stage-2 best iteration: {lgbm_reg.best_iteration_}')

# Quick valid sanity check on the soft prediction.
val_pred, val_prob, val_mag = soft_predict(lgbm_clf, lgbm_reg, X_va)
print(f'LGBM valid soft prediction: MAE={mae(y_va, val_pred):.4f}  WAPE={wape(y_va, val_pred):.4f}')


Training until validation scores don't improve for 30 rounds
Early stopping, best iteration is:
[225]	valid_0's rmse: 0.495307	valid_0's l2: 0.245329
LGBM Stage-2 best iteration: 225
LGBM valid soft prediction: MAE=6.1799  WAPE=1.7657


## Step 5 — LightGBM — test prediction + per-nest snapshot

In [10]:
lgbm_test_pred, lgbm_test_prob, lgbm_test_mag = soft_predict(lgbm_clf, lgbm_reg, X_te)
out_lgbm = test[PAIR_COLS + ['week_start', 'demand_class']].copy()
out_lgbm['y_actual'] = y_te.values
out_lgbm['y_pred']   = lgbm_test_pred
out_lgbm['s1_prob']  = lgbm_test_prob
out_lgbm['s2_mag']   = lgbm_test_mag
out_lgbm['method']   = 'LGBM_two_stage_soft'
out_lgbm.to_csv(DATA_DIR / 'lgbm_predictions.csv', index=False)

snap = (out_lgbm.groupby('NEST_NAME')
                 .apply(lambda g: pd.Series({'n': len(g),
                                             'MAE':  mae(g['y_actual'], g['y_pred']),
                                             'WAPE': wape(g['y_actual'], g['y_pred'])}))
                 .reset_index())
print('LGBM per-nest test snapshot (soft prediction):')
print(snap.to_string(index=False))
print('\nSaved lgbm_predictions.csv')


LGBM per-nest test snapshot (soft prediction):
  NEST_NAME       n      MAE     WAPE
GH1 Omenako  4240.0 2.210905 2.011639
  GH3 Vobsi  9530.0 3.025678 1.423655
RW1 Muhanga 17770.0 7.584249 1.829924
RW2 Kayonza  9910.0 8.086586 1.803490

Saved lgbm_predictions.csv


## Step 6 — XGBoost — per-demand-class feature tailoring

Each demand class (Smooth / Erratic / Intermittent / Lumpy) responds to a different set of signals. Rather than feeding all 40 features to every row, we select the most informative subset per class based on the ADI/CV² quadrant structure (Syntetos-Boylan-Croston classification):

| Class | ADI | CV² | Key signals |
|-------|-----|-----|-------------|
| **Smooth** | < 1.32 | < 0.49 | Short lags (lag 1–4), short rolling means — recent past is stable |
| **Erratic** | < 1.32 | ≥ 0.49 | All lags, all rolling incl. **std** — variability must be captured |
| **Intermittent** | ≥ 1.32 | < 0.49 | Timing features (`weeks_since_last_demand`, nonzero rates), longer lags |
| **Lumpy** | ≥ 1.32 | ≥ 0.49 | Timing + **rolling std** + max — infrequent but large, variable bursts |

One independent two-stage XGBoost is trained per class. At test time each row is routed to its class-specific model and class-specific feature slice.

In [11]:
XGB_COMMON = dict(
    n_estimators=400, learning_rate=0.05, max_depth=5,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
    random_state=42, n_jobs=-1,
)
CLASSIF_THRESHOLD = 0.35   # default; overridden per-class below
MIN_POS_FOR_STAGE2 = 30    # skip Stage-2 if fewer positive train rows

DEMAND_CLASSES = ['Smooth', 'Erratic', 'Intermittent', 'Lumpy']

# Per-class feature subsets  (all columns must exist in FEATURES / unified_features.parquet)
FEATURE_SETS = {
    # Frequent + stable: short lags reliable; long lags & intermittent timing add noise
    'Smooth': [
        'week_of_year', 'month', 'quarter', 'week_sin', 'week_cos', 'rainy_season',
        'lag_1', 'lag_2', 'lag_3', 'lag_4',
        'roll_mean_4', 'roll_std_4', 'roll_max_4', 'roll_nz_rate_4',
        'roll_mean_8', 'roll_std_8', 'roll_max_8', 'roll_nz_rate_8',
        'historical_nonzero_rate', 'cumulative_active_weeks',
        'facility_hist_mean', 'facility_hist_nz_rate',
        'product_hist_mean',  'product_hist_nz_rate',
        'fp_hist_mean',       'fp_hist_nz_rate',
        'adi', 'cv2', 'mean_nonzero', 'nonzero_weeks',
        'HEALTH_FACILITY_NAME_enc', 'PRODUCT_NAME_enc',
    ],
    # Frequent + variable: include all lags, all rolling windows + std; CV2 is high
    'Erratic': [
        'week_of_year', 'month', 'quarter', 'week_sin', 'week_cos', 'rainy_season',
        'lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_8', 'lag_12',
        'roll_mean_4',  'roll_std_4',  'roll_max_4',  'roll_nz_rate_4',
        'roll_mean_8',  'roll_std_8',  'roll_max_8',  'roll_nz_rate_8',
        'roll_mean_13', 'roll_std_13', 'roll_max_13', 'roll_nz_rate_13',
        'weeks_since_last_demand', 'last_nonzero_units',
        'historical_nonzero_rate', 'cumulative_active_weeks',
        'facility_hist_mean', 'facility_hist_nz_rate',
        'product_hist_mean',  'product_hist_nz_rate',
        'fp_hist_mean',       'fp_hist_nz_rate',
        'adi', 'cv2', 'mean_nonzero', 'nonzero_weeks',
        'HEALTH_FACILITY_NAME_enc', 'PRODUCT_NAME_enc',
    ],
    # Infrequent + stable: lag_3 unreliable (often 0); timing & nonzero-rate are primary signals
    'Intermittent': [
        'week_of_year', 'month', 'quarter', 'week_sin', 'week_cos', 'rainy_season',
        'lag_1', 'lag_2', 'lag_4', 'lag_8', 'lag_12',
        'roll_nz_rate_4', 'roll_max_4',  'roll_mean_4',
        'roll_nz_rate_8', 'roll_max_8',  'roll_mean_8',
        'roll_nz_rate_13','roll_max_13', 'roll_mean_13',
        'weeks_since_last_demand', 'last_nonzero_units',
        'historical_nonzero_rate', 'cumulative_active_weeks',
        'facility_hist_mean', 'facility_hist_nz_rate',
        'product_hist_mean',  'product_hist_nz_rate',
        'fp_hist_mean',       'fp_hist_nz_rate',
        'adi', 'cv2', 'mean_nonzero', 'nonzero_weeks',
        'HEALTH_FACILITY_NAME_enc', 'PRODUCT_NAME_enc',
    ],
    # Infrequent + variable: timing (intermittent) + magnitude variability (rolling std + max)
    'Lumpy': [
        'week_of_year', 'month', 'quarter', 'week_sin', 'week_cos', 'rainy_season',
        'lag_1', 'lag_2', 'lag_4', 'lag_8', 'lag_12',
        'roll_nz_rate_4', 'roll_max_4',  'roll_std_4',  'roll_mean_4',
        'roll_nz_rate_8', 'roll_max_8',  'roll_std_8',  'roll_mean_8',
        'roll_nz_rate_13','roll_max_13', 'roll_std_13', 'roll_mean_13',
        'weeks_since_last_demand', 'last_nonzero_units',
        'historical_nonzero_rate', 'cumulative_active_weeks',
        'facility_hist_mean', 'facility_hist_nz_rate',
        'product_hist_mean',  'product_hist_nz_rate',
        'fp_hist_mean',       'fp_hist_nz_rate',
        'adi', 'cv2', 'mean_nonzero', 'nonzero_weeks',
        'HEALTH_FACILITY_NAME_enc', 'PRODUCT_NAME_enc',
    ],
}

# Verify all features exist in the data
for dc, feats in FEATURE_SETS.items():
    missing = [f for f in feats if f not in train.columns]
    if missing:
        print(f'[WARN] {dc}: missing columns {missing}')
print('Feature counts per class:', {dc: len(f) for dc, f in FEATURE_SETS.items()})
print('Demand class distribution (train):'); print(train['demand_class'].value_counts())


Feature counts per class: {'Smooth': 32, 'Erratic': 40, 'Intermittent': 36, 'Lumpy': 39}
Demand class distribution (train):
demand_class
Intermittent    459282
Lumpy           131116
Smooth            5510
Erratic            580
Name: count, dtype: int64


## Step 7 — XGBoost — per-class Stage-1 + Stage-2 + threshold search

For each demand class we train an independent two-stage model on the rows belonging to that class. The optimal classification threshold is found by sweeping [0.20, 0.60] on the **validation** rows of the same class and picking the threshold that minimises **Asym-WAPE** (3× under-penalty). All models and their metadata are stored in `XGB_MODELS`.

In [12]:
XGB_MODELS = {}  # demand_class -> dict(clf, reg, features, threshold)

for dc in DEMAND_CLASSES:
    feats_dc = FEATURE_SETS[dc]
    tr_dc = train[train['demand_class'] == dc]
    va_dc = val[val['demand_class'] == dc]

    X_tr_dc = tr_dc[feats_dc].fillna(-999)
    y_tr_dc = tr_dc['weekly_units'].astype(float).reset_index(drop=True)
    X_va_dc = va_dc[feats_dc].fillna(-999)
    y_va_dc = va_dc['weekly_units'].astype(float).reset_index(drop=True)

    occ_tr_dc = (y_tr_dc > 0).astype(int)
    occ_va_dc = (y_va_dc > 0).astype(int)
    pos_mask_tr = (y_tr_dc > 0).to_numpy()
    pos_mask_va = (y_va_dc > 0).to_numpy()
    n_pos = int(pos_mask_tr.sum())
    n_neg = len(occ_tr_dc) - n_pos
    spw_dc = n_neg / max(n_pos, 1)

    print(f'\n── {dc} ──  train={len(tr_dc):,}  val={len(va_dc):,}  '
          f'pos_train={n_pos:,}  features={len(feats_dc)}')

    # ── Stage 1: occurrence classifier ──────────────────────────────
    clf_dc = xgb.XGBClassifier(
        **XGB_COMMON,
        objective='binary:logistic', eval_metric=['logloss', 'aucpr'],
        scale_pos_weight=spw_dc,
        early_stopping_rounds=30, verbosity=0,
    )
    clf_dc.fit(X_tr_dc, occ_tr_dc, eval_set=[(X_va_dc, occ_va_dc)], verbose=False)
    print(f'  Stage-1 best iter={clf_dc.best_iteration}')

    # ── Stage 2: magnitude regressor (positive rows only) ───────────
    reg_dc = None
    if n_pos >= MIN_POS_FOR_STAGE2 and int(pos_mask_va.sum()) >= 5:
        reg_dc = xgb.XGBRegressor(
            **XGB_COMMON,
            objective='reg:squarederror', eval_metric='rmse',
            early_stopping_rounds=30, verbosity=0,
        )
        reg_dc.fit(
            X_tr_dc[pos_mask_tr], np.log1p(y_tr_dc[pos_mask_tr]),
            eval_set=[(X_va_dc[pos_mask_va], np.log1p(y_va_dc[pos_mask_va]))],
            verbose=False,
        )
        print(f'  Stage-2 best iter={reg_dc.best_iteration}')

        # Validation soft-prediction metrics
        va_soft, _, _ = soft_predict(clf_dc, reg_dc, X_va_dc)
        upr_v, opr_v = upr_opr(y_va_dc.to_numpy(), va_soft)
        print(f'  Val (soft): WAPE={wape(y_va_dc, va_soft):.4f}  '
              f'AsymWAPE={asym_wape(y_va_dc, va_soft):.4f}  '
              f'UPR={upr_v:.4f}  OPR={opr_v:.4f}')
    else:
        print(f'  Stage-2 skipped — only {n_pos} positive rows')

    # ── Threshold search on validation ──────────────────────────────
    if reg_dc is not None:
        prob_va = clf_dc.predict_proba(X_va_dc)[:, 1]
        mag_va  = np.expm1(reg_dc.predict(X_va_dc)).clip(min=0)
        best_thr, best_aw = CLASSIF_THRESHOLD, float('inf')
        for thr in np.arange(0.20, 0.61, 0.025):
            hp = np.where(prob_va >= thr, mag_va, 0.0)
            aw = asym_wape(y_va_dc.to_numpy(), hp)
            if aw < best_aw:
                best_aw, best_thr = aw, round(float(thr), 3)
        print(f'  Best threshold: {best_thr:.3f}  (val AsymWAPE={best_aw:.4f})')
    else:
        best_thr = CLASSIF_THRESHOLD

    XGB_MODELS[dc] = dict(clf=clf_dc, reg=reg_dc,
                          features=feats_dc, threshold=best_thr)

print('\nAll classes trained:', list(XGB_MODELS.keys()))



── Smooth ──  train=5,510  val=475  pos_train=4,801  features=32
  Stage-1 best iter=35
  Stage-2 best iter=40
  Val (soft): WAPE=0.6075  AsymWAPE=0.5373  UPR=0.5022  OPR=0.1052
  Best threshold: 0.200  (val AsymWAPE=0.3772)

── Erratic ──  train=580  val=50  pos_train=490  features=40
  Stage-1 best iter=2
  Stage-2 best iter=1
  Val (soft): WAPE=0.7890  AsymWAPE=0.6758  UPR=0.6193  OPR=0.1697
  Best threshold: 0.200  (val AsymWAPE=0.5143)

── Intermittent ──  train=459,282  val=32,790  pos_train=75,650  features=36
  Stage-1 best iter=67
  Stage-2 best iter=78
  Val (soft): WAPE=2.7592  AsymWAPE=1.3506  UPR=0.6463  OPR=2.1128
  Best threshold: 0.600  (val AsymWAPE=0.9322)

── Lumpy ──  train=131,116  val=8,135  pos_train=22,768  features=39
  Stage-1 best iter=119
  Stage-2 best iter=162
  Val (soft): WAPE=1.9118  AsymWAPE=1.1271  UPR=0.7347  OPR=1.1771
  Best threshold: 0.600  (val AsymWAPE=0.9445)

All classes trained: ['Smooth', 'Erratic', 'Intermittent', 'Lumpy']


## Step 8 — XGBoost — assemble test predictions + per-nest snapshot

Each test row is routed to its demand-class model. If a class has no Stage-2 regressor (too few positive samples), all predictions for that class are zero. Both soft and hard-gated results are reported; the **soft** prediction is saved to CSV for consistency with LightGBM.

In [13]:
pieces = []
for dc in DEMAND_CLASSES:
    model = XGB_MODELS.get(dc)
    te_dc = test[test['demand_class'] == dc].copy()
    if len(te_dc) == 0:
        continue

    feats_dc = model['features']
    clf_dc   = model['clf']
    reg_dc   = model['reg']
    thr_dc   = model['threshold']
    X_te_dc  = te_dc[feats_dc].fillna(-999)

    prob_te = clf_dc.predict_proba(X_te_dc)[:, 1]
    if reg_dc is not None:
        mag_te    = np.expm1(reg_dc.predict(X_te_dc)).clip(min=0)
        pred_soft = np.clip(prob_te * mag_te, 0, None)
        pred_hard = np.where(prob_te >= thr_dc, mag_te, 0.0)
    else:
        mag_te    = np.zeros(len(te_dc))
        pred_soft = np.zeros(len(te_dc))
        pred_hard = np.zeros(len(te_dc))

    te_dc['y_pred_soft'] = pred_soft
    te_dc['y_pred_hard'] = pred_hard
    te_dc['s1_prob']     = prob_te
    te_dc['s2_mag']      = mag_te
    pieces.append(te_dc)

xgb_all = pd.concat(pieces).sort_index()

# Soft predictions — canonical output (matches LightGBM convention)
out_xgb = xgb_all[PAIR_COLS + ['week_start', 'demand_class']].copy()
out_xgb['y_actual'] = xgb_all['weekly_units'].values
out_xgb['y_pred']   = xgb_all['y_pred_soft'].values
out_xgb['s1_prob']  = xgb_all['s1_prob'].values
out_xgb['s2_mag']   = xgb_all['s2_mag'].values
out_xgb['method']   = 'XGB_per_class_soft'
out_xgb.to_csv(DATA_DIR / 'xgb_predictions.csv', index=False)

# Hard predictions — per-class threshold, saved separately for notebook 04 comparison
out_xgb_hard = out_xgb.copy()
out_xgb_hard['y_pred']  = xgb_all['y_pred_hard'].values
out_xgb_hard['method']  = 'XGB_per_class_hard'
out_xgb_hard.to_csv(DATA_DIR / 'xgb_hard_predictions.csv', index=False)

# Per-nest snapshot
def nest_snap(pred_col, label):
    rows = []
    for nest, g in xgb_all.groupby('NEST_NAME'):
        p = g[pred_col].to_numpy(); y = g['weekly_units'].to_numpy()
        u, o = upr_opr(y, p)
        rows.append({'NEST_NAME': nest, 'n': len(g),
                     'MAE':      mae(y, p),
                     'WAPE':     wape(y, p),
                     'AsymWAPE': asym_wape(y, p),
                     'UPR':      round(u, 4),
                     'OPR':      round(o, 4)})
    print(f'XGB per-nest ({label}):')
    print(pd.DataFrame(rows).to_string(index=False))

nest_snap('y_pred_soft', 'soft')
print()
nest_snap('y_pred_hard', 'hard per-class thresholds')
print('\nSaved xgb_predictions.csv  and  xgb_hard_predictions.csv')


XGB per-nest (soft):
  NEST_NAME     n       MAE     WAPE  AsymWAPE    UPR    OPR
GH1 Omenako  4240  2.992319 2.722625  1.296545 0.5835 2.1391
  GH3 Vobsi  9530  3.894117 1.832277  1.061027 0.6754 1.1569
RW1 Muhanga 17770 10.642603 2.567843  1.329459 0.7103 1.8576
RW2 Kayonza  9910 10.810772 2.411044  1.255910 0.6783 1.7327

XGB per-nest (hard per-class thresholds):
  NEST_NAME     n      MAE     WAPE  AsymWAPE    UPR    OPR
GH1 Omenako  4240 1.230519 1.119613  0.916639 0.8152 0.3045
  GH3 Vobsi  9530 2.460422 1.157688  0.926204 0.8105 0.3472
RW1 Muhanga 17770 4.753700 1.146971  0.962118 0.8697 0.2773
RW2 Kayonza  9910 5.252756 1.171482  0.959415 0.8534 0.3181

Saved xgb_predictions.csv  and  xgb_hard_predictions.csv
